# Pertes moyennes : pondérer les lots, puis accumuler les gradients

Donner à chaque exemple le poids annoncé, même quand les lots sont inégaux.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#moyennes-et-microbatches).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Le DataLoader peut terminer par un lot plus petit. Si nous journalisons seulement la moyenne de chaque lot, une moyenne de ces moyennes surreprésente ce dernier groupe.

Prenons des pertes fictives déjà moyennées : huit exemples donnent 2 ; deux exemples donnent 5. Nous connaissons ces moyennes et effectifs, pas les dix pertes individuelles.

| Lot | Perte moyenne | Exemples | Contribution |
| --- | --- | --- | --- |
| Premier | 2 | 8 | 2×8 = 16 |
| Dernier | 5 | 2 | 5×2 = 10 |
| Ensemble | 26/10 = 2,6 | 10 | 26 |

(2+5)/2 = 3,5 donne le même poids aux lots ; 2,6 donne le même poids aux exemples.

## Démonstration 1 · Reconstruire une moyenne avec les effectifs

Vérifier ce que le dénominateur représente.

### Calculer l’état suivant

Chaque couple contient une moyenne et le nombre d’exemples qu’elle représente.

In [3]:
pertes_et_tailles = [(2., 8), (5., 2)]


In [4]:
observer('pertes_et_tailles', pertes_et_tailles)

pertes_et_tailles : 2 éléments ; formes des tenseurs : []


### Calculer l’état suivant

La compréhension multiplie chaque moyenne par son effectif. Nous additionnons ensuite les deux sommes retrouvées.

In [5]:
contributions = [moyenne * taille for moyenne, taille in pertes_et_tailles]
somme = sum(contributions)


In [6]:
observer('contributions', contributions)
observer('somme', somme)

contributions = [16.0, 10.0]
somme = 26.0


### Calculer l’état suivant

Le symbole _ indique que la moyenne n’est pas utilisée dans ce calcul. Nous additionnons les tailles 8 et 2.

In [7]:
effectif = sum(taille for _, taille in pertes_et_tailles)
moyenne_globale = somme / effectif


In [8]:
observer('effectif', effectif)
observer('moyenne_globale', moyenne_globale)

effectif = 10
moyenne_globale = 2.6


### Calculer l’état suivant

La moyenne globale vaut 2,6. La moyenne des deux lots vaut 3,5 ; elle répond à une autre pondération.

In [9]:
moyenne_des_lots = sum(moyenne for moyenne, _ in pertes_et_tailles) / 2


In [10]:
observer('moyenne_des_lots', moyenne_des_lots)

moyenne_des_lots = 3.5


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [11]:
print("contributions :", contributions)
print("moyenne globale :", moyenne_globale)
print("moyenne des lots :", moyenne_des_lots)

contributions : [16.0, 10.0]
moyenne globale : 2.6
moyenne des lots : 3.5


Le bon dénominateur dépend de l’unité à laquelle nous voulons donner le même poids.

Pour une perte par position valide, il faudrait compter les positions valides, pas simplement les fichiers. L’effectif pertinent dépend du sens de la perte.

### Approfondissement — une seule mise à jour avec des microlots

Un microlot est une partie du lot traitée séparément pour limiter la mémoire. Nous pouvons accumuler ses gradients sans mettre à jour les poids entre les morceaux.

Nouvelle miniature indépendante : quatre valeurs x=[1,2,3,4] ont pour cibles 2x. Un seul poids vaut 1 et prédit poids×x. La perte est la MSE.

La perte complète vaut (1+4+9+16)/4 = 7,5. Sa dérivée vaut −15. Nous comparons le lot complet à un découpage de trois exemples puis un.

## Démonstration 2 · Comparer le gradient du tout et celui des morceaux

Garder les mêmes poids pendant tous les backward du découpage.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [12]:
import torch


### Choisir les paramètres et leur mise à jour

Le poids complet commence à 1. Le premier calcul produit une moyenne, son gradient, puis un seul pas SGD de taux 0,1.

In [13]:
entrees = torch.tensor([1., 2., 3., 4.])
cibles = 2 * entrees
poids_complet = torch.tensor(1., requires_grad=True)
optim_complet = torch.optim.SGD([poids_complet], lr=0.1)


In [14]:
observer('entrees', entrees)
observer('cibles', cibles)
observer('poids_complet', poids_complet)
observer('optim_complet', optim_complet)

entrees forme (4,) type torch.float32 requires_grad False
  valeurs : [1.0, 2.0, 3.0, 4.0]
cibles forme (4,) type torch.float32 requires_grad False
  valeurs : [2.0, 4.0, 6.0, 8.0]
poids_complet forme () type torch.float32 requires_grad True
  valeurs : 1.0


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [15]:
perte_complete = ((poids_complet * entrees - cibles) ** 2).mean()


In [16]:
observer('perte_complete', perte_complete)

perte_complete forme () type torch.float32 requires_grad True
  valeurs : 7.5


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [17]:
perte_complete.backward()


In [18]:
observer_entrainement({'entrees': entrees, 'cibles': cibles, 'poids_complet': poids_complet, 'optim_complet': optim_complet, 'perte_complete': perte_complete}, montrer_poids=False)

poids_complet.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [19]:
gradient_complet = poids_complet.grad.item()


In [20]:
observer('gradient_complet', gradient_complet)

gradient_complet = -15.0


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [21]:
optim_complet.step()



In [22]:
observer_entrainement({'entrees': entrees, 'cibles': cibles, 'poids_complet': poids_complet, 'optim_complet': optim_complet, 'perte_complete': perte_complete, 'gradient_complet': gradient_complet}, montrer_poids=True)

poids_complet forme () type torch.float32 requires_grad True
  valeurs : 2.5
poids_complet.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0


### Choisir les paramètres et leur mise à jour

Le second poids commence aussi à 1. zero_grad est appelé avant les deux morceaux, pas entre eux.

In [23]:
poids_decoupe = torch.tensor(1., requires_grad=True)
optim_decoupe = torch.optim.SGD([poids_decoupe], lr=0.1)


In [24]:
observer('poids_decoupe', poids_decoupe)
observer('optim_decoupe', optim_decoupe)

poids_decoupe forme () type torch.float32 requires_grad True
  valeurs : 1.0


### Remettre les gradients à zéro

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [25]:
optim_decoupe.zero_grad()


In [26]:
observer_entrainement({'entrees': entrees, 'cibles': cibles, 'poids_complet': poids_complet, 'optim_complet': optim_complet, 'perte_complete': perte_complete, 'gradient_complet': gradient_complet, 'poids_decoupe': poids_decoupe, 'optim_decoupe': optim_decoupe}, montrer_poids=False)

poids_complet.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0
poids_decoupe.grad = None


### Répéter la même étape sur les données

Les tranches contiennent trois exemples, puis un. Leurs coefficients sont 3/4 et 1/4, car chaque perte locale est une moyenne.

Chaque backward ajoute sa contribution dans grad. Le poids ne change qu’après la boucle ; les deux méthodes retrouvent le gradient −15.

In [27]:
for debut, fin in [(0, 3), (3, 4)]:
    x = entrees[debut:fin]
    y = cibles[debut:fin]
    perte_locale = ((poids_decoupe * x - y) ** 2).mean()
    coefficient = len(x) / len(entrees)
    contribution = coefficient * perte_locale
    contribution.backward()


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [28]:
gradient_decoupe = poids_decoupe.grad.item()


In [29]:
observer('gradient_decoupe', gradient_decoupe)

gradient_decoupe = -15.0


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [30]:
optim_decoupe.step()


In [31]:
observer_entrainement({'entrees': entrees, 'cibles': cibles, 'poids_complet': poids_complet, 'optim_complet': optim_complet, 'perte_complete': perte_complete, 'gradient_complet': gradient_complet, 'poids_decoupe': poids_decoupe, 'optim_decoupe': optim_decoupe, 'gradient_decoupe': gradient_decoupe}, montrer_poids=True)

poids_complet forme () type torch.float32 requires_grad True
  valeurs : 2.5
poids_complet.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0
poids_decoupe forme () type torch.float32 requires_grad True
  valeurs : 2.5
poids_decoupe.grad forme () type torch.float32 requires_grad False
  valeurs : -15.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [32]:
print("gradients :", gradient_complet, gradient_decoupe)
print("poids :", poids_complet.item(), poids_decoupe.item())
print("meme resultat :", torch.allclose(poids_complet, poids_decoupe))

gradients : -15.0 -15.0
poids : 2.5 2.5
meme resultat : True


Les mêmes gradients donnent ici le même pas ; des mises à jour entre les morceaux définiraient une autre trajectoire.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Un lot de huit exemples a une perte moyenne de 1 ; le dernier de deux exemples, de 3. Quelle moyenne globale ?

In [33]:
exo_tailles = [8,2]
exo_pertes = [1.,3.]
exo_moyenne = ...  # Remplacer les points par votre réponse.
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-sum(n*p for n,p in zip(exo_tailles,exo_pertes))/sum(exo_tailles)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Additionnez d’abord les contributions 8×1 et 2×3, puis divisez par dix exemples.

</details>

### Exercice 2

Quel résultat erroné donnerait une moyenne simple des deux moyennes de lots ?

In [34]:
exo_pertes = [1.,3.]
exo_moyenne_naive = ...  # Remplacer les points par votre réponse.
if exo_moyenne_naive is Ellipsis or exo_moyenne_naive is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne_naive)-sum(exo_pertes)/len(exo_pertes)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Ce calcul donne le même poids au petit lot et au grand lot. Expliquer l’erreur fait partie de l’exercice.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_tailles = [8,2]
exo_pertes = [1.,3.]
exo_moyenne = 1.4
if exo_moyenne is Ellipsis or exo_moyenne is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne)-sum(n*p for n,p in zip(exo_tailles,exo_pertes))/sum(exo_tailles)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_pertes = [1.,3.]
exo_moyenne_naive = 2.
if exo_moyenne_naive is Ellipsis or exo_moyenne_naive is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_moyenne_naive)-sum(exo_pertes)/len(exo_pertes)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [35]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()